# Environment Setup — Unity Catalog and ADLS

Run this once before the streaming notebooks. It registers the external location, creates the `retail_stream` catalog and its `landing` / `bronze` / `silver` / `gold` schemas, and an external volume that backs the streaming demos.

## External location
Point Unity Catalog at the ADLS container using a pre-created storage credential.

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS retail_stream_ext_location
URL 'abfss://retail_stream@<storage-account>.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL adls_credential)

## Catalog
Create the `retail_stream` catalog with a managed location.

In [0]:
CREATE CATALOG IF NOT EXISTS retail_stream
      MANAGED LOCATION 'abfss://retail_stream@<storage-account>.dfs.core.windows.net/'

In [0]:
use catalog retail_stream

## Landing schema

In [0]:
CREATE SCHEMA IF NOT EXISTS landing
      MANAGED LOCATION 'abfss://retail_stream@<storage-account>.dfs.core.windows.net/landing';

In [0]:
use schema landing

## Bronze / Silver / Gold schemas

In [0]:
CREATE SCHEMA IF NOT EXISTS bronze
      MANAGED LOCATION 'abfss://retail_stream@<storage-account>.dfs.core.windows.net/bronze';

      CREATE SCHEMA IF NOT EXISTS silver
      MANAGED LOCATION 'abfss://retail_stream@<storage-account>.dfs.core.windows.net/silver';

      CREATE SCHEMA IF NOT EXISTS gold
      MANAGED LOCATION 'abfss://retail_stream@<storage-account>.dfs.core.windows.net/gold';

      CREATE SCHEMA IF NOT EXISTS customers_stream
      MANAGED LOCATION 'abfss://retail_stream@<storage-account>.dfs.core.windows.net/customers_stream';
      

In [0]:
select current_catalog();

In [0]:

select current_schema();

## External volume
The streaming notebooks read source files from, and write checkpoints to, this volume.

In [0]:
create external volume if not exists operational_data
location 'abfss://retail_stream@<storage-account>.dfs.core.windows.net/landing/operational_data'

In [0]:
select * from json.`/Volumes/retail_stream/landing/operational_data/customers_stream/`

## Helper: infer a schema from a sample record
Utility (used by the streaming notebook) to derive a schema from a representative JSON record.

In [0]:
%python
from pyspark.sql.functions import schema_of_json, from_json, col

# Sample JSON (can come from a file or Kafka message)
sample_json = """
{"customer_id": 9179, "customer_name": "Alex Bennett", "date_of_birth": "1996-10-25", "telephone": "+1 6680703335", "email": "alex.bennett@example.com", "member_since": "2024-09-26", "created_timestamp": "2024-10-17 16:12:27"}
"""

# Infer schema from the sample
json_schema = spark.range(1).select(schema_of_json(sample_json).alias("schema")).collect()[0]["schema"]
print(json_schema)
